# H-002 · TSMOM clears costs (kill gate)

**Kill:** if median walk-forward **net** Sharpe is **not > 0**, STOP THE SLEEVE and do not continue to H-003.

Single-horizon daily TSMOM 12M (dataclass default) on frozen `UNIVERSE_STAR` from H-001. Do **not** freeze `HORIZON_STAR` here (that is H-003). Do **not** open 2017–2019. Do **not** re-pick the universe.

Sleeve: **core**. Primary metric: **net** `ann_sharpe`. `psr` = P(true SR > 0).


## 0. Imports & Config


In [ ]:
import os
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(ROOT, "01_data", "ingestion")):
    parent = os.path.dirname(ROOT)
    if parent == ROOT:
        break
    ROOT = parent
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from backtest.s3_trend.report import (
    fold_table,
    fold_val_metrics,
    full_is_metrics,
    load_star_stack,
    median_sharpe_hint,
    require_star,
)
from backtest.s3_trend.research import (
    RESEARCH_IS_END_S3,
    config_from_stack,
    register_hypothesis_arms,
    split_is_oos,
    star_stack_path,
)
from backtest.s3_trend.walkforward import embargo_bars_for_config, make_s3_folds
from data.processing.s3_price_panel import S3_HOLDOUT_END, price_panel_path, s3_data_dir

DATA_DIR = s3_data_dir(ROOT)
ARTIFACTS = os.path.join(ROOT, "04_backtest", "s3_trend", "artifacts")
os.makedirs(ARTIFACTS, exist_ok=True)
SLEEVE = "core"
HYP_ID = "H-002"
print("ROOT=", ROOT)
print("DATA_DIR=", DATA_DIR)
print("RESEARCH_IS_END_S3=", RESEARCH_IS_END_S3)
print("S3_HOLDOUT_END=", S3_HOLDOUT_END)


## 1. Load STAR stack


In [ ]:
STACK_PATH = star_stack_path(SLEEVE)
stack = load_star_stack(STACK_PATH)
require_star("UNIVERSE_STAR", stack.get("UNIVERSE_STAR"))
print("STAR stack path:", STACK_PATH)
print("Loaded keys with values:", {k: v for k, v in stack.items() if v is not None})
base_cfg = config_from_stack(stack)
print(base_cfg)


## 2. Data load (research IS only)


In [ ]:
def _read_parquet(path: str) -> pd.DataFrame:
    if not os.path.isfile(path):
        warnings.warn(f"missing panel: {path}")
        return pd.DataFrame()
    df = pd.read_parquet(path)
    if df.empty:
        warnings.warn(f"empty panel: {path}")
        return df
    if "date" in df.columns:
        df = df.copy()
        df["date"] = pd.to_datetime(df["date"])
    return df

price_path = price_panel_path("1d", data_dir=DATA_DIR)
panel = _read_parquet(price_path)
rates = _read_parquet(os.path.join(DATA_DIR, "g10_policy_rates.parquet"))
reer = _read_parquet(os.path.join(DATA_DIR, "bis_reer_monthly.parquet"))

if not panel.empty:
    panel_is, _holdout = split_is_oos(panel, is_end=RESEARCH_IS_END_S3)
else:
    panel_is = panel.copy()

print("price rows:", len(panel), "IS (date <=", RESEARCH_IS_END_S3.date(), "):", len(panel_is))
print("holdout 2017-2019 is sealed — not loaded for scoring here")
print("rates rows:", len(rates), "reer rows:", len(reer))

s1_weekly = None
s2_daily = None
for cand in (
    os.path.join(ROOT, "01_data", "data_files", "s1_equities", "s1_period_returns.parquet"),
    os.path.join(ROOT, "04_backtest", "s1_equities", "artifacts", "s1_period_returns.parquet"),
):
    if os.path.isfile(cand):
        s1 = pd.read_parquet(cand)
        if "date" in s1.columns:
            s1 = s1.set_index(pd.to_datetime(s1["date"]))
        num = s1.select_dtypes(include=[np.number])
        if not num.empty:
            s1_weekly = num.iloc[:, 0]
        break


## 3. Arms (pre-register)

One primary arm: daily TSMOM 12M on the frozen universe. Horizon bake-off is H-003.


In [ ]:
configs = {
    "tsmom_12m": config_from_stack(
        stack,
        hyp_overrides={
            "signal_family": "tsmom",
            "tsmom_blend": "single_12m",
            "pair_vt": False,
            "book_vt": False,
        },
    ),
}
register_hypothesis_arms(HYP_ID, list(configs.keys()), sleeve=SLEEVE, overwrite=True)
print("registered", list(configs.keys()))
print(configs["tsmom_12m"])


## 4. Fold-val + boxplots (kill gate)

Boxplots use **net** Sharpe and max DD (shown inline). If median walk-forward net Sharpe is **not > 0**, STOP THE SLEEVE. Full-IS is computed here; the locked metrics table is shown in §5.


In [ ]:
FULL_IS_COLS = [
    "arm",
    "ann_sharpe",
    "ann_sharpe_gross",
    "max_drawdown",
    "corr_to_s1",
    "corr_to_s2",
    "calmar",
    "psr",
    "dsr_local",
    "dsr_stack",
    "skew",
    "excess_kurtosis",
    "cost_bps_per_year",
    "n_days",
    "n_entries",
]

fold_df = pd.DataFrame()
full_is_df = pd.DataFrame()
_reer = reer if "reer" in dir() else pd.DataFrame()

if panel_is.empty or "date" not in panel_is.columns:
    print("SKIP fold-val / full-IS: research IS price panel missing or empty.")
    print("Run 01_data/data_files/s3_trend/s3_research_panels.ipynb first.")
else:
    is_dates = pd.DatetimeIndex(panel_is["date"].drop_duplicates().sort_values())
    try:
        folds = make_s3_folds(
            is_dates,
            n_folds=3,
            embargo_bars=embargo_bars_for_config(bar=getattr(base_cfg, "bar", "1d")),
        )
        display(fold_table(folds))
        fold_df = fold_val_metrics(
            panel_is,
            folds,
            configs,
            hyp_id=HYP_ID,
            sleeve=SLEEVE,
            s1_weekly=s1_weekly,
            s2_daily=s2_daily,
            rates_df=rates if not rates.empty else None,
            reer_df=_reer if not _reer.empty else None,
        )
        display(fold_df.head(20))

        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        if not fold_df.empty and "ann_sharpe" in fold_df.columns:
            fold_df.boxplot(column="ann_sharpe", by="arm", ax=axes[0])
            axes[0].set_title("fold-val ann_sharpe (net)")
            axes[0].set_xlabel("arm")
            axes[0].axhline(0.0, color="0.6", lw=0.8)
        if not fold_df.empty and "max_drawdown" in fold_df.columns:
            fold_df.boxplot(column="max_drawdown", by="arm", ax=axes[1])
            axes[1].set_title("fold-val max_drawdown")
            axes[1].set_xlabel("arm")
        plt.suptitle(f"{HYP_ID} fold-val")
        plt.tight_layout()
        plt.show()

        full_is_df = full_is_metrics(
            panel_is,
            configs,
            hyp_id=HYP_ID,
            sleeve=SLEEVE,
            s1_weekly=s1_weekly,
            s2_daily=s2_daily,
            rates_df=rates if not rates.empty else None,
            reer_df=_reer if not _reer.empty else None,
        )
        hint = median_sharpe_hint(fold_df)
        print("median_sharpe_hint (commentary only):", hint)
        if fold_df.empty or "ann_sharpe" not in fold_df.columns:
            print("SKIP kill gate: no fold-val net Sharpe yet.")
        else:
            med_net = float(fold_df["ann_sharpe"].median())
            print("median walk-forward net Sharpe:", med_net)
            if not (med_net > 0):
                print("STOP THE SLEEVE: median walk-forward net Sharpe is not > 0. Do not continue to H-003.")
            else:
                print("Kill gate passed: continue to H-003.")
    except Exception as exc:
        warnings.warn(f"fold-val / full-IS failed (panel may be too short): {exc!r}")


## 5. Full-IS metrics table

Locked columns: `ann_sharpe`, `ann_sharpe_gross`, `max_drawdown`, `corr_to_s1`, `corr_to_s2`, `calmar`, `psr`, `dsr_local`, `dsr_stack`, `skew`, `excess_kurtosis`, `cost_bps_per_year`, `n_days`, `n_entries`.


In [ ]:
display(full_is_df[[c for c in FULL_IS_COLS if c in full_is_df.columns]]) if not full_is_df.empty else print("no full-IS yet")


## 6. Do not freeze HORIZON_STAR

H-002 is an existence kill gate only. Horizon selection is H-003.


In [ ]:
print("H-002 does not freeze HORIZON_STAR. Leave it null until H-003.")
print("UNIVERSE_STAR=", load_star_stack(STACK_PATH).get("UNIVERSE_STAR"))
print("HORIZON_STAR=", load_star_stack(STACK_PATH).get("HORIZON_STAR"))


## 7. Sealed holdout stays closed


In [ ]:
print("Do not open 2017-2019 in H-002. Sealed holdout is a later OOS kill, not this gate.")
print("IS end", RESEARCH_IS_END_S3.date(), "holdout end", S3_HOLDOUT_END.date())
